# Librerías

In [80]:
import pandas as pd
import numpy as np
import re
import pathlib as Path
import unicodedata

# Documents

Extraer los datos de los últimos 5 años (2020 - 2024):
Hojas de formación en las empresas, contexto y errores muestreo
Extraemos todos los datos respecto a formación en las empresas pero después analizaremos solo (en el caso de algunas tablas de formación cogeremos ya las específicas de servicios [e.g. 2024 EAL-23 y 23c]):
CCAA: Total y Cataluña y otras CCAA donde tenemos sede (Madrid, Valencia, Sevilla y Bilbao)
Tamaño empresa: más de 499
Sector Transporte y almacenamiento y cuando agregado servicios

TABLAS CONTEXTO
2020, 2021, 2022, 2023, 2024:
EAL-C1, EAL-C2, EAL-C3

TABLAS FORMACIÓN EMPRESAS
2020, 2021, 2022:
EAL-16, EAL-17, EAL-18, EAL-19, EAL-20, EAL-21, EAL-22, EAL-24, EAL-25

2023, 2024 añadido:
EAL-18a, EAL-18b, EAL-18c

ERRORES MUESTREO
2020, 2021, 2022, 2023, 2024:
EAL-M1


He hagut de fer: pip install openpyxl (és l'engine de pandas per excel més nous)

In [81]:
import pandas as pd
from pathlib import Path

def encontrar_raiz_repo(inicio=None):
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")

REPO_ROOT = encontrar_raiz_repo()

# Ruta relativa dentro del repositorio Git.
file_path_rel = Path("Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx")
file_path = REPO_ROOT / file_path_rel

print("Raíz repo:", REPO_ROOT)
print("Excel origen:", file_path_rel)
print("Archivo existe:", file_path.exists())

if not file_path.exists():
    raise FileNotFoundError(f"No existe el Excel esperado: {file_path}")

sheets_to_extract = [
    # TABLAS CONTEXTO
    "EAL-C1", "EAL-C2", "EAL-C3",

    # TABLAS FORMACIÓN EMPRESAS
    "EAL-16", "EAL-17", "EAL-18", "EAL-19", "EAL-20", "EAL-21",
    "EAL-22", "EAL-23", "EAL-24", "EAL-25",
    "EAL-18a", "EAL-18b", "EAL-18c",

    # ERRORES MUESTREO
    "EAL-M1"
]

# Read selected sheets into a dictionary of DataFrames
dfs = pd.read_excel(
    file_path,
    sheet_name=sheets_to_extract,
    header=None,
    dtype=str
)

# Basic cleaning: remove fully empty rows and columns
for sheet_name, df in dfs.items():
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.reset_index(drop=True)
    dfs[sheet_name] = df


Raíz repo: /Users/fedeur/ProjecteData
Excel origen: Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx
Archivo existe: True


In [82]:
dfs["EAL-23"]

,0,1,2,3,4,5
0,ENCUESTA ANUAL LABORAL,NaN,NaN,NaN,NaN,EAL
1,,NaN,NaN,NaN,NaN,Volver al índice
2,EAL-23. EMPRESAS QUE PROPORCIONARON FORMACIÓN ...,NaN,NaN,NaN,NaN,NaN
3,Año 2024. Porcentaje sobre el total de empresa...,NaN,NaN,NaN,NaN,NaN
4,NaN,TOTAL,NADA,POCO,BASTANTE,MUCHO
5,Responder a un sistema predeterminado de promo...,100,24.833,42.321,26.86,5.987
6,Readaptar al personal a los cambios técnicos i...,100,12.615,22.193,49.421,15.771
7,Readaptar al personal a los cambios organizati...,100,14.246,29.411,44.323,12.02
8,Adaptar al personal recién incorporado a las t...,100,7.888,15.038,50.533,26.541
9,Mejorar la cualificación básica del personal p...,100,8.168,22.567,51.043,18.222


In [83]:
# Output folder dentro del repositorio Git.
output_folder_rel = Path("Equip_31/Data/Processed/EAL/2024")
output_folder = REPO_ROOT / output_folder_rel
output_folder.mkdir(parents=True, exist_ok=True)

# Save each dataframe as CSV
for sheet_name, df in dfs.items():
    output_path = output_folder / f"{sheet_name}.csv"

    df.to_csv(
        output_path,
        index=False,
        header=False,
        encoding="utf-8-sig"
    )

print(f"CSV files saved in: {output_folder_rel}")


CSV files saved in: Equip_31/Data/Processed/EAL/2024


## Normalización estructural de EAL-16

A partir de aquí se trabaja con el CSV extraído de la hoja `EAL-16`. Las rutas se resuelven desde la raíz del repositorio para que el notebook funcione aunque se ejecute desde otra carpeta.


In [84]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

from pathlib import Path
import re
import unicodedata
import pandas as pd

def encontrar_raiz_repo(inicio=None):
    """Devuelve la raíz del repositorio buscando la carpeta Equip_31."""
    ruta = Path(inicio or Path.cwd()).resolve()
    for candidata in [ruta, *ruta.parents]:
        if (candidata / "Equip_31").exists():
            return candidata
    raise FileNotFoundError("No se encontró la raíz del repositorio con Equip_31.")

REPO_ROOT = encontrar_raiz_repo()

# Ruta relativa dentro del repositorio Git.
archivo_csv_rel = Path("Equip_31/Data/Processed/EAL/2024/EAL-16.csv")
archivo_csv = REPO_ROOT / archivo_csv_rel

# Excel oficial raw, usado solo para comprobación y trazabilidad.
archivo_raw_rel = Path("Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx")
archivo_raw = REPO_ROOT / archivo_raw_rel

ANIO = 2024
HOJA = "EAL-16"

print("Raíz repo:", REPO_ROOT)
print("Ruta relativa:", archivo_csv_rel)
print("CSV existe:", archivo_csv.exists())
print("CSV resuelto:", archivo_csv)
print("Raw existe:", archivo_raw.exists())
print("Raw resuelto:", archivo_raw)

if not archivo_csv.exists():
    raise FileNotFoundError(f"No existe el CSV esperado: {archivo_csv}")
if not archivo_raw.exists():
    raise FileNotFoundError(f"No existe el Excel raw esperado: {archivo_raw}")


Raíz repo: /Users/fedeur/ProjecteData
Ruta relativa: Equip_31/Data/Processed/EAL/2024/EAL-16.csv
CSV existe: True
CSV resuelto: /Users/fedeur/ProjecteData/Equip_31/Data/Processed/EAL/2024/EAL-16.csv
Raw existe: True
Raw resuelto: /Users/fedeur/ProjecteData/Equip_31/Data/raw/EAL/Tablas_EAL_2024.xlsx


## Funciones auxiliares

Estas funciones limpian texto, detectan las subtablas internas de `EAL-16` y asignan metadatos estructurales: ámbito, sector, tamaño de empresa y CCAA.


In [85]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def limpiar_texto(valor):
    if pd.isna(valor):
        return ""
    valor = str(valor).replace("\n", " ")
    valor = re.sub(r"\s+", " ", valor)
    return valor.strip()

def quitar_acentos(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()

def extraer_codigo_subtabla(texto):
    match = re.match(r"^(EAL-16[a-f]?)\.", limpiar_texto(texto), flags=re.IGNORECASE)
    return match.group(1).upper() if match else None

def metadatos_subtabla(codigo, titulo):
    """
    Asigna solo las dimensiones que realmente desagrega cada subtabla.

    Regla metodológica para EAL-16:
    - EAL-16: total empresas; no desagrega por sector, tamaño ni CCAA.
    - EAL-16a/b/c: desagrega por tamaño de empresa; sector y CCAA no aplican.
    - EAL-16d/e/f: desagrega por sector agregado; tamaño y CCAA no aplican.
    """
    metadatos = {
        "Ámbito": "Total empresas",
        "Sector": "",
        "Tamaño Empresa": "",
        "CCAA": "",
    }

    if codigo == "EAL-16A":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = "5 a 49 trabajadores"
    elif codigo == "EAL-16B":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = "50 a 499 trabajadores"
    elif codigo == "EAL-16C":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = "Más de 499 trabajadores"
    elif codigo == "EAL-16D":
        metadatos["Ámbito"] = "Sector agregado"
        metadatos["Sector"] = "Industria"
    elif codigo == "EAL-16E":
        metadatos["Ámbito"] = "Sector agregado"
        metadatos["Sector"] = "Construcción"
    elif codigo == "EAL-16F":
        metadatos["Ámbito"] = "Sector agregado"
        metadatos["Sector"] = "Servicios"

    return metadatos

def fila_es_cabecera(row):
    valores = [limpiar_texto(x).upper() for x in row.tolist()]
    return {"TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"}.issubset(set(valores))


## Lectura del CSV original

Se lee el CSV procesado por la extracción previa, sin asumir encabezado fijo.


In [86]:
# ============================================================
# LECTURA DEL CSV ORIGINAL
# ============================================================

df_raw = pd.read_csv(
    archivo_csv,
    header=None,
    dtype=str,
    keep_default_na=False
)

df_raw = df_raw.map(limpiar_texto)
df_raw = df_raw.replace("", pd.NA)
df_raw = df_raw.dropna(how="all")
df_raw = df_raw.dropna(axis=1, how="all")
df_raw = df_raw.fillna("").reset_index(drop=True)

print("Dimensión original:", df_raw.shape)
display(df_raw.head(20))


Dimensión original: (93, 6)


,0,1,2,3,4,5
0,ENCUESTA ANUAL LABORAL,,,,,EAL
1,,,,,,Volver al índice
2,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PA...,,,,,
3,Año 2024. Porcentaje sobre el total de empresas.,,,,,
4,,TOTAL,NADA,POCO,BASTANTE,MUCHO
5,De dirección,100,10.414,18.818,40.283,30.484
6,De trabajo en equipo,100,2.224,5.822,45.125,46.829
7,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336
8,Administrativas de oficina,100,11.117,25.313,37.788,25.782
9,De resolución de problemas (localización de pr...,100,5.893,15.752,44.063,34.292


## EAL-16 en formato ancho

La salida solicitada es una fila por categoría y bloque, manteniendo las columnas originales de respuesta (`TOTAL`, `NADA`, `POCO`, `BASTANTE`, `MUCHO`). En `EAL-16` hay 7 bloques y 10 categorías por bloque, por tanto quedan 70 registros.

In [87]:
# ============================================================
# EAL-16 A FORMATO ANCHO
# ============================================================

COLUMNAS_EAL16 = ["TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"]

registros = []
titulo_actual = None
codigo_actual = None
columnas_actuales = None
metadatos_actuales = None

for _, row in df_raw.iterrows():
    primera_celda = limpiar_texto(row.iloc[0])
    codigo_detectado = extraer_codigo_subtabla(primera_celda)

    if codigo_detectado:
        codigo_actual = codigo_detectado
        titulo_actual = primera_celda
        columnas_actuales = None
        metadatos_actuales = metadatos_subtabla(codigo_actual, titulo_actual)
        continue

    if codigo_actual and fila_es_cabecera(row):
        columnas_actuales = [limpiar_texto(x).upper() for x in row.tolist()]
        continue

    if not (codigo_actual and columnas_actuales):
        continue

    categoria = primera_celda
    if categoria == "":
        continue

    registro = {
        "Año": ANIO,
        "Hoja": HOJA,
        "Título de tabla": titulo_actual,
        "Categoría": categoria,
    }

    for posicion, columna in enumerate(columnas_actuales):
        if columna in COLUMNAS_EAL16:
            registro[columna] = pd.to_numeric(limpiar_texto(row.iloc[posicion]), errors="coerce")

    registros.append(registro)

df_eal16_ancho = pd.DataFrame(registros)
columnas_eal16_ancho = [
    "Año", "Hoja", "Título de tabla", "Categoría",
    "TOTAL", "NADA", "POCO", "BASTANTE", "MUCHO"
]
df_eal16_ancho = df_eal16_ancho[columnas_eal16_ancho]

assert df_eal16_ancho.shape[0] == 70
assert df_eal16_ancho[COLUMNAS_EAL16].notna().all().all()

print("EAL-16 ancho hasta columna MUCHO:", df_eal16_ancho.shape)
print("Esperado: 7 bloques x 10 categorías = 70 registros")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal16_ancho)


EAL-16 ancho hasta columna MUCHO: (70, 9)
Esperado: 7 bloques x 10 categorías = 70 registros


,Año,Hoja,Título de tabla,Categoría,TOTAL,NADA,POCO,BASTANTE,MUCHO
0,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De dirección,100,10.414,18.818,40.283,30.484
1,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De trabajo en equipo,100,2.224,5.822,45.125,46.829
2,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,De atención al público/ trato a clientes,100,5.128,13.392,37.144,44.336
3,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,Administrativas de oficina,100,11.117,25.313,37.788,25.782
4,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,"De resolución de problemas (localización de problemas o fallos, análisis de sus causas y búsqueda de soluciones)",100,5.893,15.752,44.063,34.292
5,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,En lenguas extranjeras,100,23.022,40.356,24.568,12.054
6,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,Básicas de cálculo y/o comunicación oral o escrita,100,22.688,36.513,29.697,11.102
7,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,Generales de tecnologías de la información,100,13.593,29.346,39.173,17.888
8,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,Profesionales de tecnologías de la información,100,26.908,37.656,24.493,10.944
9,2024,EAL-16,EAL-16. EMPRESAS SEGÚN GRADO DE IMPORTANCIA PARA EL DESARROLLO DE LA EMPRESA EN LOS PRÓXIMOS AÑOS DE DIVERSAS COMPET...,"Competencias técnicas, prácticas y otras específicas del puesto de trabajo",100,13.417,18.861,36.496,31.226


## EAL-17 y EAL-18 en formato ancho

Para estas hojas se mantiene una fila por categoría/desglose y se conservan las columnas originales de la tabla. La columna `Ámbito` indica si la fila corresponde a total, tamaño de empresa, sector o CCAA.

In [88]:
# ============================================================
# FUNCIONES PARA EAL-17 Y EAL-18 EN FORMATO ANCHO
# ============================================================

SECCIONES_FILA = {
    "TAMAÑO DE LA EMPRESA": "Tamaño empresa",
    "ACTIVIDAD ECONÓMICA": "Sector",
    "COMUNIDAD AUTÓNOMA": "CCAA",
}


def leer_csv_hoja(hoja):
    ruta_rel = Path(f"Equip_31/Data/Processed/EAL/2024/{hoja}.csv")
    ruta = REPO_ROOT / ruta_rel
    if not ruta.exists():
        raise FileNotFoundError(f"No existe el CSV esperado: {ruta}")

    df = pd.read_csv(ruta, header=None, dtype=str, keep_default_na=False)
    df = df.map(limpiar_texto)
    df = df.replace("", pd.NA)
    df = df.dropna(how="all")
    df = df.dropna(axis=1, how="all")
    df = df.fillna("").reset_index(drop=True)
    return df, ruta_rel


def metadatos_fila(categoria, seccion_actual):
    metadatos = {
        "Ámbito": "Total empresas",
        "Sector": "",
        "Tamaño Empresa": "",
        "CCAA": "",
    }

    if categoria.upper() == "TOTAL":
        return metadatos

    if seccion_actual == "Tamaño empresa":
        metadatos["Ámbito"] = "Tamaño empresa"
        metadatos["Tamaño Empresa"] = categoria
    elif seccion_actual == "Sector":
        metadatos["Ámbito"] = "Sector"
        metadatos["Sector"] = categoria
    elif seccion_actual == "CCAA":
        metadatos["Ámbito"] = "CCAA"
        metadatos["CCAA"] = categoria

    return metadatos


def detectar_titulo_csv(df, hoja):
    for valor in df.iloc[:, 0].tolist():
        texto = limpiar_texto(valor)
        if texto.upper().startswith(f"{hoja}."):
            return texto
    return ""


def nombres_columnas_eal17(df):
    grupo_superior = [limpiar_texto(x) for x in df.iloc[4].tolist()]
    subvariable = [limpiar_texto(x) for x in df.iloc[5].tolist()]

    columnas = {}
    grupo_actual = ""
    for col in range(1, df.shape[1]):
        if grupo_superior[col]:
            grupo_actual = grupo_superior[col]
        sub = subvariable[col]

        if col == 1:
            columnas[col] = "TOTAL"
        elif grupo_actual and sub:
            columnas[col] = f"{grupo_actual} - {sub}"
        elif grupo_actual:
            columnas[col] = grupo_actual
        else:
            columnas[col] = sub
    return columnas, 6


def nombres_columnas_eal18(df):
    grupo_superior = [limpiar_texto(x) for x in df.iloc[4].tolist()]
    subvariable = [limpiar_texto(x) for x in df.iloc[5].tolist()]

    columnas = {}
    grupo_actual = ""
    for col in range(1, df.shape[1]):
        if grupo_superior[col]:
            grupo_actual = grupo_superior[col]
        sub = subvariable[col]
        columnas[col] = f"{grupo_actual} - {sub}" if grupo_actual and sub else (grupo_actual or sub)
    return columnas, 6


def parsear_eal_17_18_ancho(hoja, constructor_columnas):
    df, ruta_rel = leer_csv_hoja(hoja)
    titulo = detectar_titulo_csv(df, hoja)
    columnas_valor, fila_inicio_datos = constructor_columnas(df)

    registros = []
    seccion_actual = None

    for _, row in df.iloc[fila_inicio_datos:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "":
            continue

        categoria_norm = categoria.upper()
        if categoria_norm in SECCIONES_FILA:
            seccion_actual = SECCIONES_FILA[categoria_norm]
            continue
        if categoria.startswith("("):
            continue

        metadatos = metadatos_fila(categoria, seccion_actual)
        registro = {
            "Año": ANIO,
            "Hoja": hoja,
            "Título de tabla": titulo,
            "Categoría": categoria,
            "Ámbito": metadatos["Ámbito"],
            "Sector": metadatos["Sector"],
            "Tamaño Empresa": metadatos["Tamaño Empresa"],
            "CCAA": metadatos["CCAA"],
        }

        for col, nombre_columna in columnas_valor.items():
            valor = limpiar_texto(row.iloc[col]) if col < len(row) else ""
            registro[nombre_columna] = pd.to_numeric(valor, errors="coerce")

        registros.append(registro)

    df_ancho = pd.DataFrame(registros)
    columnas_base = ["Año", "Hoja", "Título de tabla", "Categoría"]
    columnas_dim = ["Ámbito", "Sector", "Tamaño Empresa", "CCAA"]
    columnas_valores = [nombre for _, nombre in sorted(columnas_valor.items())]
    df_ancho = df_ancho[columnas_base + columnas_valores + columnas_dim]

    print(f"{hoja} desde {ruta_rel}: {df.shape} -> {df_ancho.shape}")
    return df_ancho


## EAL-17 en formato ancho

In [89]:
# ============================================================
# EAL-17 A FORMATO ANCHO
# ============================================================

df_eal17_ancho = parsear_eal_17_18_ancho("EAL-17", nombres_columnas_eal17)

assert df_eal17_ancho.shape[0] == 32
print("Esperado EAL-17: 32 filas de categorías/desgloses")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal17_ancho.iloc[:, :10])


EAL-17 desde Equip_31/Data/Processed/EAL/2024/EAL-17.csv: (42, 7) -> (32, 14)
Esperado EAL-17: 32 filas de categorías/desgloses


,Año,Hoja,Título de tabla,Categoría,TOTAL,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Total,EMPRESAS QUE PROPORCIONAN FORMACIÓN - Impartición de cursos y otros tipos de formación (1),EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo impartición de cursos (1),EMPRESAS QUE PROPORCIONAN FORMACIÓN - Sólo otros tipos de formación (1),EMPRESAS QUE NO PROPORCIONAN FORMACIÓN
0,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",TOTAL,100,73.468,60.220,25.366,14.415,26.532
1,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 5 a 9 trabajadores,100,65.319,51.321,28.014,20.665,34.681
2,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 10 a 49 trabajadores,100,79.228,64.481,25.163,10.356,20.772
3,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 50 a 249 trabajadores,100,94.759,79.559,15.091,5.349,5.241
4,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",De 250 a 499 trabajadores,100,98.371,85.856,11.710,2.434,1.629
5,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",Más de 499 trabajadores,100,99.298,88.305,9.742,1.953,0.702
6,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",Industria,100,76.222,64.223,20.911,14.866,23.778
7,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",Construcción,100,77.713,61.593,29.467,8.940,22.287
8,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",Comercio y reparación de vehículos,100,73.959,62.349,23.788,13.863,26.041
9,2024,EAL-17,"EAL-17. EMPRESAS SEGÚN SI IMPARTIERON FORMACIÓN POR TAMAÑO DE LA EMPRESA, ACTIVIDAD ECONÓMICA Y COMUNIDAD AUTÓNOMA",Transporte y almacenamiento,100,77.405,55.376,34.796,9.828,22.595


## EAL-18 en formato ancho

In [90]:
# ============================================================
# EAL-18 A FORMATO ANCHO
# ============================================================

df_eal18_ancho = parsear_eal_17_18_ancho("EAL-18", nombres_columnas_eal18)

assert df_eal18_ancho.shape[0] == 32
print("Esperado EAL-18: 32 filas de categorías/desgloses")
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal18_ancho.iloc[:, :10])


EAL-18 desde Equip_31/Data/Processed/EAL/2024/EAL-18.csv: (43, 7) -> (32, 14)
Esperado EAL-18: 32 filas de categorías/desgloses


,Año,Hoja,Título de tabla,Categoría,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Total,DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Proporcionaron formación (1),DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - No proporcionaron formación (1),NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Total,NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - Proporcionaron formación (2),NO DETECTARON NECESIDADES FORMATIVAS DE SU PERSONAL - No proporcionaron formación (2)
0,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,TOTAL,29.123694,89.974511,10.025489,70.876306,66.570590,33.429410
1,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 5 a 9 trabajadores,23.671862,83.544726,16.455274,76.328138,59.481181,40.518819
2,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 10 a 49 trabajadores,31.724690,93.100741,6.899259,68.275310,72.749236,27.250764
3,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 50 a 249 trabajadores,48.701094,98.742720,1.266525,51.298906,90.863612,9.136388
4,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,De 250 a 499 trabajadores,58.708189,100.000000,0.000000,41.253364,96.831314,3.168686
5,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,Más de 499 trabajadores,62.753623,99.769053,0.230947,37.246377,98.702983,1.297017
6,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,Industria,29.340514,91.943067,8.063373,70.659486,69.761720,30.238280
7,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,Construcción,27.053957,90.611641,9.388359,72.946043,73.016647,26.983353
8,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,Comercio y reparación de vehículos,28.467349,90.868950,9.131050,71.532651,67.384332,32.615668
9,2024,EAL-18,EAL-18. EMPRESAS SEGÚN DETECTARAN O NO NECESIDADES FORMATIVAS DE SU PERSONAL E IMPARTIERAN O NO FORMACIÓN POR TAMAÑO...,Transporte y almacenamiento,25.901905,95.031299,4.968701,74.098095,71.218545,28.781455


## EAL-18a, EAL-18b, EAL-18c, EAL-19 y EAL-20 en formato ancho simple

Se continúa con el mismo criterio: una fila por categoría/desglose y las columnas originales de valores. Por ahora no se añaden columnas de ámbito, sector, tamaño empresa ni CCAA al final.

In [91]:
# ============================================================
# FUNCIONES PARA FORMATO ANCHO SIMPLE
# ============================================================

MARCADORES_SECCION = {
    "TAMAÑO DE LA EMPRESA",
    "ACTIVIDAD ECONÓMICA",
    "COMUNIDAD AUTÓNOMA",
}


def titulo_hoja(df, hoja):
    for valor in df.iloc[:, 0].tolist():
        texto = limpiar_texto(valor)
        if texto.upper().startswith(f"{hoja}."):
            return texto
    return ""


def limpiar_nombre_columna(nombre):
    nombre = limpiar_texto(nombre)
    nombre = re.sub(r"\s+", " ", nombre)
    return nombre


def hacer_columnas_unicas_simple(columnas):
    resultado = []
    contador = {}
    for columna in columnas:
        columna = limpiar_nombre_columna(columna) or "SIN_TITULO"
        if columna not in contador:
            contador[columna] = 1
            resultado.append(columna)
        else:
            contador[columna] += 1
            resultado.append(f"{columna}_{contador[columna]}")
    return resultado


def parsear_hoja_ancha_simple(hoja, fila_cabecera, fila_inicio_datos):
    df, ruta_rel = leer_csv_hoja(hoja)
    titulo = titulo_hoja(df, hoja)
    columnas_valor = hacer_columnas_unicas_simple(df.iloc[fila_cabecera, 1:].tolist())

    registros = []
    for _, row in df.iloc[fila_inicio_datos:].iterrows():
        categoria = limpiar_texto(row.iloc[0])
        if categoria == "":
            continue
        if categoria.upper() in MARCADORES_SECCION:
            continue
        if categoria.startswith("("):
            continue

        registro = {
            "Año": ANIO,
            "Hoja": hoja,
            "Título de tabla": titulo,
            "Categoría": categoria,
        }

        for offset, columna in enumerate(columnas_valor, start=1):
            valor = limpiar_texto(row.iloc[offset]) if offset < len(row) else ""
            registro[columna] = pd.to_numeric(valor, errors="coerce")

        # Evita filas sin ningún dato numérico.
        if all(pd.isna(registro[col]) for col in columnas_valor):
            continue

        registros.append(registro)

    df_ancho = pd.DataFrame(registros)
    df_ancho = df_ancho[["Año", "Hoja", "Título de tabla", "Categoría", *columnas_valor]]
    print(f"{hoja} desde {ruta_rel}: {df.shape} -> {df_ancho.shape}")
    return df_ancho


## EAL-18a en formato ancho

EAL-18a: medios utilizados por empresas que proporcionaron formación.

In [92]:
# ============================================================
# EAL-18a A FORMATO ANCHO SIMPLE
# ============================================================

df_eal18a_ancho = parsear_hoja_ancha_simple("EAL-18a", fila_cabecera=4, fila_inicio_datos=5)

print("EAL-18a ancho:", df_eal18a_ancho.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal18a_ancho)


EAL-18a desde Equip_31/Data/Processed/EAL/2024/EAL-18a.csv: (41, 7) -> (32, 10)
EAL-18a ancho: (32, 10)


,Año,Hoja,Título de tabla,Categoría,TOTAL,Cursos de formación diseñados y gestionados por su empresa,Cursos de formación diseñados y gestionados por otra organización,"Formación planificada en el puesto de trabajo, utilizando los medios habituales de trabajo","Aprendizaje planificado a partir de rotación de puestos de trabajo, intercambios, etc.","Participación en conferencias, seminarios, grupos de trabajo, talleres o ferias de muestras"
0,2024,EAL-18a,,TOTAL,73.468278,34.269329,73.709969,56.005492,27.375278,35.086542
1,2024,EAL-18a,,De 5 a 9 trabajadores,65.319423,27.217848,65.989117,51.141371,22.937813,32.188656
2,2024,EAL-18a,,De 10 a 49 trabajadores,79.227905,35.028637,79.002872,57.331022,29.277350,33.998708
3,2024,EAL-18a,,De 50 a 249 trabajadores,94.759353,58.777973,84.135506,70.508861,36.475507,51.399249
4,2024,EAL-18a,,De 250 a 499 trabajadores,98.385236,76.162564,85.384916,76.865963,46.697929,57.796014
5,2024,EAL-18a,,Más de 499 trabajadores,99.275362,85.450122,84.963504,80.048662,51.532847,65.596107
6,2024,EAL-18a,,Industria,76.222600,34.293577,76.121179,61.910901,36.081017,33.867169
7,2024,EAL-18a,,Construcción,77.712290,22.420652,84.768610,57.771672,21.273003,23.030251
8,2024,EAL-18a,,Comercio y reparación de vehículos,73.958727,38.680615,71.445779,54.400700,29.940268,41.624315
9,2024,EAL-18a,,Transporte y almacenamiento,77.406769,30.568829,79.668783,52.052104,24.258690,20.416312


## EAL-18b en formato ancho

EAL-18b: formación durante suspensión de contrato o reducción de jornada.

In [93]:
# ============================================================
# EAL-18b A FORMATO ANCHO SIMPLE
# ============================================================

df_eal18b_ancho = parsear_hoja_ancha_simple("EAL-18b", fila_cabecera=5, fila_inicio_datos=7)

print("EAL-18b ancho:", df_eal18b_ancho.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal18b_ancho)


EAL-18b desde Equip_31/Data/Processed/EAL/2024/EAL-18b.csv: (41, 9) -> (31, 12)
EAL-18b ancho: (31, 12)


,Año,Hoja,Título de tabla,Categoría,Empresas que realizaron suspensión de contrato/reducción de jornada,Total,El 0'%,Más del 0% hasta el 25%,Más del 25% hasta el 50%,Más del 50% hasta el 75%,Más del 75% hasta el 100%,El 100%
0,2024,EAL-18b,,De 5 a 9 trabajadores,0.763845,100,85.635,9.391,1.418,2.351,0.000,1.206
1,2024,EAL-18b,,De 10 a 49 trabajadores,1.563983,100,82.378,13.232,0.000,2.052,0.312,2.025
2,2024,EAL-18b,,De 50 a 249 trabajadores,3.367701,100,64.167,23.950,4.927,1.052,1.926,3.978
3,2024,EAL-18b,,De 250 a 499 trabajadores,9.919262,100,69.215,19.402,4.646,0.000,3.834,2.904
4,2024,EAL-18b,,Más de 499 trabajadores,11.787440,100,66.136,26.049,5.791,1.364,0.325,0.334
5,2024,EAL-18b,,Industria,3.526077,100,79.042,12.156,0.385,1.863,0.546,6.008
6,2024,EAL-18b,,Construcción,0.131222,100,84.067,8.563,0.058,5.920,1.391,0.000
7,2024,EAL-18b,,Comercio y reparación de vehículos,0.230648,100,85.276,11.029,1.964,0.000,0.000,1.732
8,2024,EAL-18b,,Transporte y almacenamiento,0.851236,100,82.853,16.628,0.328,0.000,0.191,0.000
9,2024,EAL-18b,,Hostelería,2.805053,100,86.534,13.319,0.128,0.019,0.000,0.000


## EAL-18c en formato ancho

EAL-18c: formación en seguridad y salud laboral.

In [94]:
# ============================================================
# EAL-18c A FORMATO ANCHO SIMPLE
# ============================================================

df_eal18c_ancho = parsear_hoja_ancha_simple("EAL-18c", fila_cabecera=4, fila_inicio_datos=6)

print("EAL-18c ancho:", df_eal18c_ancho.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal18c_ancho)


EAL-18c desde Equip_31/Data/Processed/EAL/2024/EAL-18c.csv: (40, 2) -> (31, 5)
EAL-18c ancho: (31, 5)


,Año,Hoja,Título de tabla,Categoría,PORCENTAJE DE EMPRESAS QUE PROPORCIONARON FORMACIÓN EN SEGURIDAD Y SALUD LABORAL
0,2024,EAL-18c,,De 5 a 9 trabajadores,68.611
1,2024,EAL-18c,,De 10 a 49 trabajadores,81.343
2,2024,EAL-18c,,De 50 a 249 trabajadores,92.156
3,2024,EAL-18c,,De 250 a 499 trabajadores,95.577
4,2024,EAL-18c,,Más de 499 trabajadores,96.501
5,2024,EAL-18c,,Industria,81.214
6,2024,EAL-18c,,Construcción,81.458
7,2024,EAL-18c,,Comercio y reparación de vehículos,74.164
8,2024,EAL-18c,,Transporte y almacenamiento,81.207
9,2024,EAL-18c,,Hostelería,73.552


## EAL-19 en formato ancho

EAL-19: empresas que proporcionaron formación por tamaño y sector agregado.

In [95]:
# ============================================================
# EAL-19 A FORMATO ANCHO SIMPLE
# ============================================================

df_eal19_ancho = parsear_hoja_ancha_simple("EAL-19", fila_cabecera=4, fila_inicio_datos=5)

print("EAL-19 ancho:", df_eal19_ancho.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal19_ancho)


EAL-19 desde Equip_31/Data/Processed/EAL/2024/EAL-19.csv: (11, 5) -> (6, 8)
EAL-19 ancho: (6, 8)


,Año,Hoja,Título de tabla,Categoría,TOTAL,INDUSTRIA,CONSTRUCCIÓN,SERVICIOS
0,2024,EAL-19,EAL-19. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,TOTAL,73.468278,76.222600,77.712290,72.148643
1,2024,EAL-19,EAL-19. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,De 5 a 9 trabajadores,65.319423,63.058146,71.260095,64.621839
2,2024,EAL-19,EAL-19. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,De 10 a 49 trabajadores,79.227905,80.668769,83.545151,78.058545
3,2024,EAL-19,EAL-19. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,De 50 a 249 trabajadores,94.759353,97.169168,95.046440,93.740024
4,2024,EAL-19,EAL-19. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,De 250 a 499 trabajadores,98.385236,99.467377,100.000000,97.818599
5,2024,EAL-19,EAL-19. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN SECTOR DE ACTIVIDAD POR TAMAÑO DE LA EMPRESA,Más de 499 trabajadores,99.275362,99.782609,98.717949,99.151436


## EAL-20 en formato ancho

EAL-20: competencias en las que se formó por tamaño de empresa.

In [96]:
# ============================================================
# EAL-20 A FORMATO ANCHO SIMPLE
# ============================================================

df_eal20_ancho = parsear_hoja_ancha_simple("EAL-20", fila_cabecera=4, fila_inicio_datos=5)

print("EAL-20 ancho:", df_eal20_ancho.shape)
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", 120):
    display(df_eal20_ancho)


EAL-20 desde Equip_31/Data/Processed/EAL/2024/EAL-20.csv: (17, 7) -> (11, 10)
EAL-20 ancho: (11, 10)


,Año,Hoja,Título de tabla,Categoría,TOTAL,DE 5 A 9 TRABAJADORES,DE 10 A 49 TRABAJADORES,DE 50 A 249 TRABAJADORES,DE 250 A 499 TRABAJADORES,MÁS DE 499 TRABAJADORES
0,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,De dirección,16.347340,15.343619,15.804993,22.430750,25.322392,30.510949
1,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,De trabajo en equipo,28.494848,29.855895,26.931117,28.578895,34.153966,33.333333
2,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,De atención al público/ trato a clientes,24.530267,27.373940,22.586210,19.846059,21.570926,27.055961
3,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,Administrativas de oficina,17.027644,15.670464,18.738422,15.816981,12.817507,13.333333
4,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,Técnicas específicas del puesto de trabajo,50.838810,48.254094,52.131239,56.943983,55.763970,54.014599
5,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,"De resolución de problemas (localización de problemas o fallos, análisis de sus causas y búsqueda de soluciones)",11.209969,11.767293,10.401264,11.911436,15.787417,13.187348
6,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,En lenguas extranjeras,7.478712,3.941978,7.740614,21.100394,27.471669,27.639903
7,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,Generales de tecnologías de la información,12.666991,11.324888,12.758545,17.261367,21.492771,25.109489
8,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,Profesionales de tecnologías de la información,5.310919,3.812620,5.996873,8.690075,7.854631,12.749392
9,2024,EAL-20,EAL-20. EMPRESAS QUE PROPORCIONARON FORMACIÓN A SUS TRABAJADORES SEGÚN TAMAÑO DE LA EMPRESA POR COMPETENCIAS EN LAS ...,Básicas de cálculo y/o comunicación oral o escrita,1.254936,1.059013,1.173541,2.537179,2.110199,2.822384
